In [ ]:
import moku_util as mu

import scipy.optimize as optimize

# The moku_util module loads these libraries for internal methods,
# but you may want to run alternative code that uses them directly.
import matplotlib.pyplot as plt
import numpy as np

#### Load the data and look at some metadata as well as the raw traces

In [ ]:
# Acquired data from both 'slow' channels of the SiPM readout, to serve
# as example. You'll probably want to acquire your own data and change
# the filename appropriately.
#
# Note that the file itself is too large to be hosted on GitHub, but 
# is available on Canvas for download. You shouldn't need to use it
# since you can acquire your own data, but it's there.
filename = '../example_data/cs137_2channel_pulses.li'
df = mu.DataFile(filename)

In [ ]:
# Note that the sampling rate may not correspond exactly to the 
# requested sampling rate. The Moku:Go may adjust the sampling 
# rate given limitations of the hardware, but it should be close
# to the requested value.
print(f'    Sampling rate : {df.fsamp} Hz')
print(f'Number of samples : {df.nsamp}')

#### Plot the data and zoom in on various features to see if the logged data matches your expectation

Compare what you see to what you've been observing on the oscilloscope.

In [ ]:
%matplotlib widget
fig, ax = df.plot_data(input=[0,1], microseconds=True, xlim=(0,10000))

#### Make sure to close the figure for the later parts of this notebook!

You won't be able to interact with the figure and zoom in etc if you close the figure prematurely, so hopefully you didn't just use the "Run All" feature of the notebook...

In [ ]:
plt.close(fig)

---
#### The function executed from below scapes the data and finds peaks above baseline

Data from the peak finding, such as peak locations etc are stored as class attributes. The raw output from `scipy.signal.find_peaks()` is stored, as well as some derived attributes. 

There are some default values, but you should change these based on your own observations of the baseline and peak height.

In [ ]:
# The relevant input parameters are:
#     - the channel to analyze
#     - the minimum height of a peak, i.e. a threshold value
#     - the minimum width of a peak in units of samples
#     - the prominence of a peak, i.e. the necessary height
#       of the peak above baseline to be considered as a peak.
#       The tuple is for minimum and maximum prominence, although
#       the maximum is not used in this example.
#     - a boolean to specify whether to look for negative peaks,
#       where all the other inputs should still remain positive
#       but be interpreted as negative values.
df.find_peaks(
    channel=0, 
    height=0.5, 
    width=2, 
    prominence=(0.3, None),
    negative_signal=False
)

Note that this class method adds the peak data as a class attribute, so you can run it again on a second input channel. There is an optional boolean argument for negative pulses if you're analyzing the 'fast' channel.

---
#### Now let's make a first pass at analysis

There's a lot we can do now that we've collected data and identified all of the pulses contained in it. Let's start by thinking about the timing, and we'll move on to energy later.

First, let's look at the pulse rate and compare that to the known activity of the source

In [ ]:
# Determine the total number of pulses detected in the first channel,
# which is "Input 1" on the Moku:Go, but index 0 in the code. Determine
# the total integration time of the file by analyzing the endpoints.
npulses = len(df.peaks[0])
total_file_time = df.time[-1] - df.time[0]

print(f"Detected {npulses} pulses in {total_file_time:.1f} seconds")
print(f"Estimated pulse rate: {npulses/total_file_time:.2f} Hz.")

Compare your detected pulse rate to the source activity

What is the activity of the source in decays/second? How does this compare to the rate that you've measured? If there's a discrepancy, can you explain it?

---
#### How many pulses do you 'count' per unit time?

Thinking of decays as rare events, are there any distributions you know of that might described how pulses are distributed in time?

We can analyze this question using a histogram of pulse times!

In [ ]:
# Extract the peak times from the DataFile object, by indexing the time
# array with the peak indices.
peak_times = df.time[df.peaks[0]]

# Generate a histogram of the peak times, dividing the total integration
# time bins of length delta_t.
delta_t = 0.2    # Time window to consider
nbins = int(total_file_time/delta_t)  # Number of bins to cover the total time
bin_edges = np.linspace(0, total_file_time, nbins+1)

# Plot the data and extract the histogram values in one go
fig, ax = plt.subplots(figsize=(6,4))
vals, _, _ = ax.hist(peak_times, bins=bin_edges)
ax.set_xlabel('Time (s)')
ax.set_ylabel(f'Number of events per {delta_t:.2g} s')
fig.tight_layout()
plt.show()

print(f'    Average number of events per {delta_t:.2g} s : {np.mean(vals):.2f}')
print(f'Standard deviation of events per {delta_t:.2g} s : {np.std(vals):.2f}')

Change the `delta_t` value a few times and record the results. Does it match your expectations for a 'counting' experiment? Think back to Ph79L when we were counting muons.

If it doesn't _exactly_ match your expectations, do you have an explanation for why that might be? Can you acquire more data or analyze it differently to test your prediction?

---
#### What about the time between pulses?

On larger timescales, it appears that pulses are uniformly distributed in time, although with some fluctuations governed by counting statistics. Let's analyze the time-like structure of these fluctuations.

Consider that the probability of an individual nucleus to undergo decay is determined by an exponential function $p \propto e^{-t/\tau}$, where $\tau = \frac{t_{1/2}}{\ln 2}$, with $t_{1/2}$ being the halflife. This means that the time between successive events should also follow an exponential distribution, given that the individual decays are independent events. The time constant is different, but we'll get to that later.

In [ ]:
# Compute the time between consecutive peaks.
time_diffs = np.diff(peak_times)

fig, ax = plt.subplots(figsize=(6,4))
tdiff_vals, tdiff_bin_edges, _ = ax.hist(time_diffs, bins=100)
ax.set_xlabel('Time between consecutive peaks (s)')
ax.set_ylabel('Number of events')
# ax.set_yscale('log') # Plot with and without log scale
fig.tight_layout()
plt.show()

#### Let's test if our data does indeed follow the expected distribution

We'll fit the histogram of `time_diffs` to an exponential

In [ ]:
def exponential(t, A, tau):
    return A * np.exp(-t/tau)

In [ ]:
# Generate the time points corresponding to the histogram values
# (midpoints of the bins) and optionally mask out points below
# a certain time difference as part of the fit. Default value
# doesn't mask any points to start.
time_points = (tdiff_bin_edges[:-1] + tdiff_bin_edges[1:])/2
# mask = (time_points < total_file_time) & (time_points > 0)
mask = (time_points < 0.004) & (time_points > 0.0005)

# Guess the initial time constant and amplitude to help
# the fit converge. Do this by examining your histogram.
param_guess = [6000, 0.001]

# Fit the exponential function to the histogram data, making
# use of scipy's curve_fit function. The popt variable contains the
# optimal fit parameters, and pcov contains the covariance matrix of
# the fit parameters, which can be used to estimate uncertainties.
popt, pcov = \
    optimize.curve_fit(
        exponential, 
        time_points[mask], 
        tdiff_vals[mask],
        p0=param_guess
    )

#### Interpreting our fit

The time constant $\tau$ can be thought of as the inverse of a _rate parameter_, since $\tau$ represents the mean value of the time differences between consecutive events. How does your fit result compare to your earlier measured activity level?

In [ ]:
rate_estimate = 1.0 / popt[1]
print(f'Estimated rate from exponential fit: {rate_estimate:.2f} Hz.')

For a more robust explanation, consider that with some approximately constant activity level $A$ in events/sec, the probability of one decay happening in some time $dt$ is given by $P \approx A \, dt$. 

Thus the probability for _no decay_ to happen in some time $t$ from an arbitrary starting point can be written as $P \approx (1 - A \, dt)^{t/dt}$, where the time interval from $0$ to $t$ has been split up into many short segments each of length $dt$, each with probability $A \, dt$ of an decay happening.

Taking the limit as $dt \rightarrow 0$, we would find $P = e^{-A t}$

---
#### Let's plot our data, now with the fit

We'll also include residuals, as well as an indication of the range of values actually used in the regression, which may be less than all of the data if the mask parameter was used.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(6,5), height_ratios=[3,1], sharex=True)

ax[0].hist(time_diffs, bins=100)
ax[0].plot(time_points[mask], exponential(time_points[mask], *popt), 'r--', lw=2)
ax[0].set_ylabel('Number of events')

residuals = tdiff_vals - exponential(time_points, *popt)
ax[1].scatter(time_points, residuals)
ax[1].axhline(0, color='k', ls='--')
ax[1].axvline(np.min(time_points[mask]), color='r', ls='--', lw=2)
ax[1].axvline(np.max(time_points[mask]), color='r', ls='--', lw=2)
ax[1].set_xlabel('Time between consecutive peaks (s)')
ax[1].set_ylabel('Residuals')

ax[0].set_yscale('log') # Plot with and without log scale
fig.tight_layout()
plt.show()

#### How 'good' is the fit? The cell below may fail or produce a warning when you first run it!

Consider how reduced $\chi^2$ is calculated while examining your histogram of time differences between successive events. How could we modify the code to estimate $\chi^2$ without encountering the error? If you have an idea for how to do it conceptually, but don't know how to implement it in Python, don't hesitate to ask!

In [ ]:
# Generate a mask to exclude points outside some range of interest.
# You may also want to use this same mask during the regression itself
# mask = (time_points < total_file_time) & (time_points > 0)
mask = (time_points < 0.004) & (time_points > 0.0005)

# Compute the residuals between the data and the fit, and calculate
# the chi-squared statistic.
residuals = tdiff_vals[mask] - exponential(time_points[mask], *popt)
undertainties = np.sqrt(tdiff_vals[mask])  # Assuming counting statistics
chi2 = np.sum(residuals**2 / undertainties**2)

# Generate a reduced chi-squared statistic by dividing out the degrees
# of freedom, which is the number of data points minus the number of fit
# parameters.
ndof = len(tdiff_vals[mask]) - len(popt)
reduced_chi2 = chi2 / ndof

print(f'Goodness of fit: reduced chi^2 = {reduced_chi2:.2f}')

Based on the value of the reduced $\chi^2$ that you find, do counting statistics accurately capture the fluctuations in our data? 

Discuss and explain.

---
#### If you finish everything above for one channel, try comparing data from multiple channels by re-running the code with a different channel selection at the initial step. What would you expect to be different?

#### You may also wish to collect more raw data with the Moku as part of this process